# PCA & Dimensionality Reduction

**Dimensionality reduction** = squeezing many features into fewer while keeping most of the information.

**PCA (Principal Component Analysis)** finds new axes ('principal components') pointing in the directions of most variation, and keeps only the top few.

We'll take a 30-feature dataset and reduce it to 2 dimensions we can actually plot. PCA is **unsupervised** — it never looks at the labels.

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: Load 30-dimensional data

30 features = impossible to visualize directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target
print(f'Original shape: {X.shape}  ({X.shape[1]} features — can\'t plot this!)')

## Step 2: Scale the features FIRST (important for PCA)

PCA looks at variance, so features must be on comparable scales. We standardize each feature to mean 0, std 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

## Step 3: Apply PCA to reduce 30D -> 2D

Same fit/transform pattern. We ask for 2 components.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

print(f'Reduced shape: {X_2d.shape}  (30 features -> 2 components)')
print(f'Variance captured by PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'Variance captured by PC2: {pca.explained_variance_ratio_[1]:.1%}')
print(f'Total kept in just 2 components: {pca.explained_variance_ratio_.sum():.1%}')

`explained_variance_ratio_` tells you how much information each component keeps. Often 2 components keep a surprising % of the total — that's the power of PCA.

## Step 4: Now we can SEE the 30D data in 2D

Each point is a tumor, colored by its true class. PCA used NO labels — yet the classes separate visibly. That means the structure was really there in the data.

In [ ]:
plt.figure(figsize=(8, 6))
for class_value, color, name in [(0, 'red', data.target_names[0]), (1, 'green', data.target_names[1])]:
    mask = y == class_value
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=name, alpha=0.6)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('30 features squeezed into 2 (PCA)')
plt.legend()
plt.grid(True)
plt.show()

## Step 5: How many components do you need?

Plot cumulative variance to decide. Often you can keep 95% of the info with far fewer than all 30 features.

In [ ]:
pca_full = PCA().fit(X_scaled)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)

plt.plot(range(1, len(cumulative) + 1), cumulative, marker='o')
plt.axhline(0.95, color='red', linestyle='--', label='95% information')
plt.xlabel('Number of components')
plt.ylabel('Cumulative variance explained')
plt.title('How many components to keep?')
plt.legend()
plt.grid(True)
plt.show()

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f'Just {n_95} components keep 95% of the information (vs all 30 features).')

## Your turn

1. Change `n_components=2` to `3` and check the total variance captured. How much more do you gain?
2. From Step 5, how many components capture 99% of the information?
3. Train a `LogisticRegression` on the 2D PCA data vs the full 30D data. How much accuracy do you lose by using only 2 features? (Often surprisingly little!)
4. In your own words: why is PCA called 'unsupervised', and why does it help with visualization and speed?